# Vilier Colab Runner


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/ngocbao220/vilier.git /content/vilier
%cd /content/vilier


In [ ]:
DIARIZATION_MODEL_CHOICES = [
    "pyannote/speaker-diarization-community-1",
    "pyannote/speaker-diarization-3.1",
]
DIARIZATION_MODEL = DIARIZATION_MODEL_CHOICES[0]
PYANNOTE_PROFILE = "pyannote-community"

if DIARIZATION_MODEL == "pyannote/speaker-diarization-3.1":
    PYANNOTE_PROFILE = "pyannote-3.1"

!python -m pip install -r requirements/{PYANNOTE_PROFILE}.txt
!python -m pip install -r requirements/sepreformer.txt


In [ ]:
import json
import os
from pathlib import Path

DRIVE_AUDIO_PATH = "/content/drive/MyDrive/VDT-TurnTaking/inputs/real.wav"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/VDT-TurnTaking/outputs"
PIPELINE_DEVICE = "auto"
ENABLE_OVERLAP_SEPARATION = True
OVERLAP_SEPARATION_MODEL = "SepReformer_Base_WSJ0"
SEPREFORMER_CHECKPOINT_REPO = "niobures/SepReformer"

config_path = Path("config.colab.json")
audio_path = Path(DRIVE_AUDIO_PATH)
output_dir = Path(DRIVE_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

config = json.loads(Path("config.json").read_text(encoding="utf-8"))
config["entrypoint"]["input_path"] = str(audio_path)
config["entrypoint"]["output_path"] = str(output_dir)
config["diarization"]["backend"] = "pyannote"
config["diarization"]["model"] = DIARIZATION_MODEL
config["diarization"]["token_env"] = "HUGGINGFACE_TOKEN"
config["diarization"]["device"] = PIPELINE_DEVICE
config["overlap_separation"]["enabled"] = bool(ENABLE_OVERLAP_SEPARATION)
config["overlap_separation"]["backend"] = "sepreformer"
config["overlap_separation"]["model_name"] = OVERLAP_SEPARATION_MODEL
config["overlap_separation"]["checkpoint_repo"] = SEPREFORMER_CHECKPOINT_REPO
config["overlap_separation"]["checkpoint_revision"] = ""
config["overlap_separation"]["device"] = PIPELINE_DEVICE
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["CONFIG_PATH"] = str(config_path)
os.environ["INPUT_PATH"] = str(audio_path)
os.environ["OUTPUT_PATH"] = str(output_dir)


In [ ]:
!bash run.sh
